# TAAF demo workflow

Walk through the framework end to end: build a couple of `Benchmark`s, run them, then generate a cross-run comparison HTML so you can compare two solvers side by side.

Each benchmark plays a list of `Game`s with a chosen `Solver`, `n_passes` times. `Benchmark.run()` writes its own `diagnostics.html` (R4.01), `summary.txt` (R4.02), and `benchmark.json` (R2.13) under `job_dir`, and prints both the summary and the path to its HTML when it finishes.

After the runs finish, the only thing this notebook actually generates is the **comparison HTML** that ties two runs together (R4.11/R4.12). All outputs go under `notebooks/runs/demo/` (gitignored).

Spec context: [`docs/requirements.md`](../docs/requirements.md) — implementation status: [`docs/requirement_status.md`](../docs/requirement_status.md).

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

import taaf

out_root = Path("runs/demo")
out_root.mkdir(parents=True, exist_ok=True)

## Run two benchmarks

Both runs play the same two games for `n_passes=2`, and they only differ in their solver:

- `ExampleGame` — toy two-level "press the right sequence" game.
- `GameAPI(env_name="ls20")` — real `arc_agi` env. Uses the default `ArcadeSpec()` — `OperationMode.OFFLINE` against the env files bundled with `re-arc-3` (no network, no API key). Pass `arcade_spec=ArcadeSpec(operation_mode=arc_agi.OperationMode.ONLINE, ...)` to hit the live ARC-AGI3 API instead.

Solvers:

- **`seq_winner`** — `SolverSequence` driving a hand-crafted sequence: 3× ACTION1 + 3× ACTION2 wins `ExampleGame`; a RESET separator and the 13-move solution from `arc-agi-framework`'s test suite (`[3,3,3,1,1,1,1,4,4,4,1,1,1]`) clears the first level of `ls20`.
- **`random_seed42`** — `SolverRandom` with `seed=42`, capped at 20 actions per game. Reproducible (same RNG seed → same trajectory) but doesn't beeline anywhere.

When `bm.run()` finishes it prints both the text summary and the path to its `diagnostics.html` — click those if you want to drill into either run on its own.

In [ ]:
import arcengine

# ARC-AGI3 maps each move to an `arcengine.GameAction` (RESET, ACTION1, ...,
# ACTION7). The numeric ids 1 and 2 are what ExampleGame's two levels expect.
def act(action_id: int):
    return arcengine.ActionInput(id=arcengine.GameAction.from_id(action_id), data={})

# 6-move solution to ExampleGame (3× ACTION1 wins level 0, 3× ACTION2 wins level 1),
# followed by a RESET separator, followed by the 13-move solution to ls20's first
# level (from `arc-agi-framework/tests/test_game.py`). SolverSequence plays the
# whole thing on every game; ExampleGame finishes after move 6 and ignores the
# rest, while ls20 absorbs the prefix, RESETs back to a clean state, and then
# clears its first level.
ls20_level0_winning_moves = [3, 3, 3, 1, 1, 1, 1, 4, 4, 4, 1, 1, 1]
combined_winning_sequence = (
    [act(1)] * 3 + [act(2)] * 3       # ExampleGame
    + [act(0)]                         # RESET separator
    + [act(i) for i in ls20_level0_winning_moves]
)

games: list[taaf.game.Game] = [
    taaf.game_examples.ExampleGame(),
    taaf.game_api.GameAPI(env_name="ls20"),  # default ArcadeSpec() → OFFLINE re-arc-3
]

bm_a = taaf.benchmark.Benchmark(
    label="seq_winner",
    games=games,
    solver=taaf.solver_examples.SolverSequence(actions=combined_winning_sequence),
    n_passes=2,
    job_dir=out_root / "seq_winner",
)
bm_b = taaf.benchmark.Benchmark(
    label="random_seed42",
    games=games,
    solver=taaf.solver_examples.SolverRandom(seed=42, max_actions_per_game=20),
    n_passes=2,
    job_dir=out_root / "random_seed42",
)
await bm_a.run()
await bm_b.run()

# And if you want to deploy:
# taaf.deploy.kaggle(bm_a) -> Kaggle notebook created, ready to run the specified games, test submission, or run submission (will slot in test games).
# taaf.deploy.tufa_cluster(bm_a, 'b200') -> SLURM job queued on Tufa B200 cluster.
# taaf.deploy.inline(bm_a) -> runs inline in my current environment, but with the same isolation and logging as if it were deployed on the cluster (R2.35-R2.37).

## Build the comparison HTML

Per-run HTML / summary / JSON were already written by `Benchmark.run()`. The only thing missing for a side-by-side view is the cross-run comparison: paired t-test, head-to-head per-game scatter, and per-game drill-down pages. `taaf.diagnostics.generate_comparison_html` produces all of that under `notebooks/runs/demo/comparison/`.

The cell below also displays a clickable list of the comparison entrypoint.

In [ ]:
comparison_dir = out_root / "comparison"
taaf.diagnostics.generate_comparison_html([bm_a, bm_b], comparison_dir)

display(Markdown(f"- [**Comparison HTML**]({comparison_dir}/index.html)"))